1. Incremental load for index data

In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

# Define your stock mappings: (Ticker symbol, Bronze table name)
stocks_config = [
    ("^GSPC", "bronze.sp500_initial"),
    ("^STOXX", "bronze.stoxx_eu_initial"),
    ("^N225", "bronze.nikkei_initial"),
    ("000001.SS", "bronze.ssec_initial"),
    ("^NSEI", "bronze.nsei_initial")
]

for ticker, table_name in stocks_config:
    print(f"Processing next single-year incremental load for {ticker} into {table_name}...")
    
    # 1. Find the maximum Date present in the Bronze table
    max_date_row = spark.sql(f"SELECT MAX(CAST(Date AS DATE)) AS max_date FROM {table_name}").collect()
    latest_loaded_date = max_date_row[0]["max_date"]
    
    if latest_loaded_date is None:
        raise ValueError(f"No existing dates found in {table_name}. Please run the initial load first.")
        
    # 2. Determine the next target year based on the latest recorded date
    target_year = latest_loaded_date.year + 1
    
    # 3. Set start and exclusive end dates for that exact single calendar year
    start_date = f"{target_year}-01-01"
    end_date = f"{target_year}-12-31" # yfinance 'end' parameter is exclusive
    
    print(f"Fetching data for target year {target_year} ({start_date} to {end_date})...")
    
    # 4. Fetch data for the single year using yfinance
    new_df = yf.download(ticker, start=start_date, end=end_date)
    
    if new_df.empty:
        print(f"No market data returned for {ticker} for the year {target_year}.\n")
        continue
        
    # 5. Reset index so 'Date' becomes a standard column
    new_df = new_df.reset_index()
    
    # 6. Flatten columns if yfinance returns a multi-index structure
    if isinstance(new_df.columns, pd.MultiIndex):
        new_df.columns = new_df.columns.get_level_values(0)
        
    # 7. Convert to Spark DataFrame
    spark_new_df = spark.createDataFrame(new_df)
    
    # 8. Add ingestion timestamp
    spark_new_df = spark_new_df.withColumn("ingestion_timestamp", F.current_timestamp())
    
    # 9. Append the single year's records to the Bronze table
    (
        spark_new_df.write.format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(table_name)
    )
    
    print(f"Successfully appended year {target_year} records for {ticker} to {table_name}!\n")

print("Single-year incremental stock load completed successfully!")